In [724]:
import pandas as pd                                                                     # type: ignore
import numpy as np                                                                      # type: ignore

### Importing dataset

In [725]:
import sys
sys.path.insert(1, "/home/mendrika/mendrika-phd/codes/nflics")
import nflics  

In [726]:
location = "Dakar"
data = pd.read_csv(f"/home/mendrika/mendrika-phd/data/other-location/cleaned/train-data-{location}.csv", index_col=False)
test_data = pd.read_csv(f"/home/mendrika/mendrika-phd/data/other-location/cleaned/test-data-{location}.csv", index_col=False)

#### combining dataset and doing some exploratory analysis

In [727]:
data = data.dropna()
test_data = test_data.dropna()

In [728]:
dataset = "train"
if dataset == "train":
    data = data
else:
    data = test_data

In [729]:
#data = data.drop([f"Cb_{location.lower()}_t2",	f"Cb_{location.lower()}_t3"], axis=1)

In [730]:
data['datetime'] = pd.to_datetime(data[['year', 'month', 'day', 'hour', 'minute']])

In [731]:
data = data.sort_values(by='datetime').reset_index(drop=True)

In [732]:
def find_exact_row_after_given_hours(row, hours, df):
    target_time = row['datetime'] + pd.Timedelta(hours=hours)
    corresponding_row = df[df['datetime'] == target_time]
    if not corresponding_row.empty:
        return corresponding_row.index[0]  # Return the index of the corresponding row
    else:
        return None

In [733]:
lead_time = 7
given_hours = lead_time - 1
data['row_index_Cb'] = data.apply(find_exact_row_after_given_hours, args=(given_hours, data), axis=1)
data['row_index_X0'] = data.apply(find_exact_row_after_given_hours, args=(1, data), axis=1)

In [734]:
data

,year,month,day,hour,minute,lat1,lon1,lat2,lon2,lat3,...,size5,ds1,ds2,ds3,ds4,ds5,Cb_Dakar_t1,datetime,row_index_Cb,row_index_X0
0,2004,6,1,17,45,12.08,-8.22,10.06,-8.13,14.00,...,0.0,325.14,352.41,430.00,430.00,430.00,0.0,2004-06-01 17:45:00,NaN,4.0
1,2004,6,1,18,0,12.13,-8.18,10.02,-8.31,14.00,...,0.0,326.53,347.12,430.00,430.00,430.00,0.0,2004-06-01 18:00:00,24.0,5.0
2,2004,6,1,18,15,12.17,-8.18,10.02,-8.45,14.00,...,0.0,326.27,343.59,430.00,430.00,430.00,0.0,2004-06-01 18:15:00,25.0,6.0
3,2004,6,1,18,30,12.00,-8.94,12.13,-8.27,14.00,...,0.0,301.39,323.64,430.00,430.00,430.00,0.0,2004-06-01 18:30:00,26.0,7.0
4,2004,6,1,18,45,12.04,-8.85,10.60,-8.18,14.00,...,0.0,304.60,343.29,430.00,430.00,430.00,0.0,2004-06-01 18:45:00,27.0,8.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
103872,2019,9,30,5,0,12.62,-16.04,12.08,-17.03,13.34,...,0.0,72.47,83.35,91.27,109.44,430.00,0.0,2019-09-30 05:00:00,NaN,103875.0
103873,2019,9,30,5,15,13.39,-15.72,12.76,-16.04,12.13,...,11800.0,71.06,79.61,89.20,105.99,123.49,0.0,2019-09-30 05:15:00,NaN,103876.0
103874,2019,9,30,5,30,12.00,-16.49,12.22,-17.07,14.00,...,0.0,86.21,97.27,430.00,430.00,430.00,0.0,2019-09-30 05:30:00,NaN,NaN
103875,2019,9,30,6,0,11.41,-17.07,12.31,-16.98,14.00,...,0.0,83.60,115.07,430.00,430.00,430.00,0.0,2019-09-30 06:00:00,NaN,NaN


In [735]:
#data = data.rename(columns={'Cb_Dakar':'Cb_Dakar_t1'})

In [736]:
data_specific_column = data[['row_index_Cb']].copy()
data_specific_column[f'Cb_{location.lower()}_t{lead_time}'] = data_specific_column['row_index_Cb'].apply(lambda x: data.loc[x, f'Cb_{location}_t1'] if pd.notna(x) else None)

data = data.drop([f"Cb_{location}_t1", "datetime"], axis=1)

# Merge the specific column back to the original DataFrame
data = data.merge(data_specific_column[[f'Cb_{location.lower()}_t{lead_time}']], left_index=True, right_index=True)

In [737]:
data = data.drop(["row_index_Cb"], axis=1).dropna()

In [738]:
data = data.merge(data.drop(columns=['row_index_X0', f'Cb_{location.lower()}_t{lead_time}']), left_on='row_index_X0', right_index=True, suffixes=('', '_t1'))

In [739]:
data = data.drop(columns=['row_index_X0'])

In [740]:
column_to_move = data.pop(f'Cb_{location.lower()}_t{lead_time}')
data[f'Cb_{location.lower()}_t{lead_time}'] = column_to_move

In [741]:
data = data.dropna()

In [742]:
data.to_csv(f"/home/mendrika/mendrika-phd/data/other-location/cleaned/{dataset}-data-{location}-lt{lead_time}-with-t-1.csv", index=False)